# 🔎 MLflow Tracking — Experiment Exploration & Model Loading

This notebook demonstrates how to:

1. 📋 **Retrieve all runs** from an MLflow experiment by name
2. 📊 **Inspect metrics, parameters, and tags** for each run
3. 📦 **Load a model** using a specific Run ID
4. 🔢 **Check model version**, parameters, and metrics
5. ✅ **Make predictions** with the loaded model

---

In [1]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from pprint import pprint

/opt/anaconda3/envs/mlenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---
## 1️⃣ Set the Experiment Name & Retrieve Experiment Info

In [2]:
# ──────────────────────────────────────────────
# Define the experiment name
# ──────────────────────────────────────────────
EXPERIMENT_NAME = "RF model versioning demo"

# Get the experiment by name
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

if experiment is None:
    print(f"❌ Experiment '{EXPERIMENT_NAME}' not found!")
    print("Make sure you have run the training notebook first.")
else:
    print("=" * 80)
    print("✅ EXPERIMENT FOUND")
    print("=" * 80)
    print(f"Experiment Name:    {experiment.name}")
    print(f"Experiment ID:      {experiment.experiment_id}")
    print(f"Artifact Location:  {experiment.artifact_location}")
    print(f"Lifecycle Stage:    {experiment.lifecycle_stage}")
    print(f"Creation Time:      {experiment.creation_time}")
    print(f"Last Updated:       {experiment.last_update_time}")

✅ EXPERIMENT FOUND
Experiment Name:    RF model versioning demo
Experiment ID:      1
Artifact Location:  /Users/jogeswararao/ML_AI_TRAINING_SESSIONS/CODE_DATA_MODEL_VERSIONING_IN_ML_LIFECYCLE/mlruns/1
Lifecycle Stage:    active
Creation Time:      1790944756105
Last Updated:       1790944756105


---
## 2️⃣ List All Runs in the Experiment

In [3]:
# ──────────────────────────────────────────────
# Search all runs in the experiment
# ──────────────────────────────────────────────
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=['start_time DESC']
)

print("=" * 80)
print(f"📋 TOTAL RUNS FOUND: {len(runs)}")
print("=" * 80)

# Display key columns for all runs
display_cols = ['run_id', 'experiment_id', 'status', 'start_time', 'end_time']

# Add metric columns if present
metric_cols = [col for col in runs.columns if col.startswith('metrics.')]
param_cols = [col for col in runs.columns if col.startswith('params.')]
tag_cols = [col for col in runs.columns if col.startswith('tags.')]

print(f"\nMetric columns found:    {metric_cols}")
print(f"Parameter columns found: {param_cols}")
print(f"Tag columns found:       {[t for t in tag_cols if not t.startswith('tags.mlflow')]}")

runs[display_cols + metric_cols]

📋 TOTAL RUNS FOUND: 1

Metric columns found:    ['metrics.test_accuracy', 'metrics.cv_accuracy', 'metrics.test_precision', 'metrics.test_f1_score', 'metrics.test_recall']
Parameter columns found: ['params.classifier__min_samples_split', 'params.classifier__max_depth', 'params.classifier__n_estimators', 'params.classifier__min_samples_leaf']
Tag columns found:       ['tags.pipeline', 'tags.dataset', 'tags.tuning_method', 'tags.model_type']


,run_id,experiment_id,status,start_time,end_time,metrics.test_accuracy,metrics.cv_accuracy,metrics.test_precision,metrics.test_f1_score,metrics.test_recall
0,e92368539ff94b94a7f3d6356f2b3759,1,FINISHED,2026-10-02 13:19:55.488000+00:00,2026-10-02 13:19:58.886000+00:00,0.999905,0.99986,0.999905,0.999905,0.999905


In [4]:
# ──────────────────────────────────────────────
# Detailed Run Summary Table
# ──────────────────────────────────────────────
print("=" * 80)
print("📊 DETAILED RUN SUMMARY")
print("=" * 80)

for idx, row in runs.iterrows():
    print(f"\n{'─' * 60}")
    print(f"Run #{idx + 1}")
    print(f"{'─' * 60}")
    print(f"  Run ID:      {row['run_id']}")
    print(f"  Status:      {row['status']}")
    print(f"  Start Time:  {row['start_time']}")
    print(f"  End Time:    {row['end_time']}")
    
    # Print run name if available
    if 'tags.mlflow.runName' in row and pd.notna(row['tags.mlflow.runName']):
        print(f"  Run Name:    {row['tags.mlflow.runName']}")
    
    # Print metrics
    if metric_cols:
        print(f"  ── Metrics ──")
        for mc in metric_cols:
            if pd.notna(row[mc]):
                metric_name = mc.replace('metrics.', '')
                print(f"    {metric_name}: {row[mc]:.4f}")
    
    # Print params
    if param_cols:
        print(f"  ── Parameters ──")
        for pc in param_cols:
            if pd.notna(row[pc]):
                param_name = pc.replace('params.', '')
                print(f"    {param_name}: {row[pc]}")
    
    # Print custom tags
    custom_tags = [t for t in tag_cols if not t.startswith('tags.mlflow')]
    if custom_tags:
        print(f"  ── Tags ──")
        for tc in custom_tags:
            if pd.notna(row[tc]):
                tag_name = tc.replace('tags.', '')
                print(f"    {tag_name}: {row[tc]}")

📊 DETAILED RUN SUMMARY

────────────────────────────────────────────────────────────
Run #1
────────────────────────────────────────────────────────────
  Run ID:      e92368539ff94b94a7f3d6356f2b3759
  Status:      FINISHED
  Start Time:  2026-10-02 13:19:55.488000+00:00
  End Time:    2026-10-02 13:19:58.886000+00:00
  Run Name:    best_rf_pipeline
  ── Metrics ──
    test_accuracy: 0.9999
    cv_accuracy: 0.9999
    test_precision: 0.9999
    test_f1_score: 0.9999
    test_recall: 0.9999
  ── Parameters ──
    classifier__min_samples_split: 2
    classifier__max_depth: None
    classifier__n_estimators: 50
    classifier__min_samples_leaf: 1
  ── Tags ──
    pipeline: preprocessor + classifier
    dataset: online_retail
    tuning_method: GridSearchCV
    model_type: RandomForestClassifier


---
## 3️⃣ Load Model from a Specific Run ID

In [5]:
# ──────────────────────────────────────────────
# Specify the Run ID to load
# ──────────────────────────────────────────────
RUN_ID = "e92368539ff94b94a7f3d6356f2b3759"

print(f"🎯 Target Run ID: {RUN_ID}")

🎯 Target Run ID: e92368539ff94b94a7f3d6356f2b3759


In [6]:
# ──────────────────────────────────────────────
# Load the model using the Run ID
# ──────────────────────────────────────────────
model_uri = f"runs:/{RUN_ID}/best_rf_pipeline_model"

print(f"Loading model from: {model_uri}")
loaded_model = mlflow.sklearn.load_model(model_uri)

print(f"\n✅ Model loaded successfully!")
print(f"Model type: {type(loaded_model).__name__}")
print(f"\n── Pipeline Steps ──")
for step_name, step_obj in loaded_model.steps:
    print(f"  {step_name}: {type(step_obj).__name__}")

Loading model from: runs:/e92368539ff94b94a7f3d6356f2b3759/best_rf_pipeline_model



✅ Model loaded successfully!
Model type: Pipeline

── Pipeline Steps ──
  preprocessor: ColumnTransformer
  classifier: RandomForestClassifier


---
## 4️⃣ Inspect Run Details — Metrics, Parameters & Model Version

In [7]:
# ──────────────────────────────────────────────
# Get the full run details from MLflow
# ──────────────────────────────────────────────
run_data = mlflow.get_run(RUN_ID)

print("=" * 80)
print("📋 RUN INFORMATION")
print("=" * 80)
print(f"Run ID:          {run_data.info.run_id}")
print(f"Run Name:        {run_data.info.run_name}")
print(f"Experiment ID:   {run_data.info.experiment_id}")
print(f"Status:          {run_data.info.status}")
print(f"Start Time:      {pd.Timestamp(run_data.info.start_time, unit='ms')}")
print(f"End Time:        {pd.Timestamp(run_data.info.end_time, unit='ms')}")
print(f"Artifact URI:    {run_data.info.artifact_uri}")
print(f"Lifecycle Stage: {run_data.info.lifecycle_stage}")

📋 RUN INFORMATION
Run ID:          e92368539ff94b94a7f3d6356f2b3759
Run Name:        best_rf_pipeline
Experiment ID:   1
Status:          FINISHED
Start Time:      2026-10-02 13:19:55.488000
End Time:        2026-10-02 13:19:58.886000
Artifact URI:    /Users/jogeswararao/ML_AI_TRAINING_SESSIONS/CODE_DATA_MODEL_VERSIONING_IN_ML_LIFECYCLE/mlruns/1/e92368539ff94b94a7f3d6356f2b3759/artifacts
Lifecycle Stage: active


In [8]:
# ──────────────────────────────────────────────
# Inspect Metrics
# ──────────────────────────────────────────────
metrics = run_data.data.metrics

print("=" * 80)
print("📊 METRICS")
print("=" * 80)

metrics_df = pd.DataFrame({
    'Metric': list(metrics.keys()),
    'Value': list(metrics.values())
}).sort_values('Metric').reset_index(drop=True)

for _, row in metrics_df.iterrows():
    print(f"  {row['Metric']:25s} → {row['Value']:.4f}")

print(f"\nTotal metrics logged: {len(metrics)}")
metrics_df

📊 METRICS
  cv_accuracy               → 0.9999
  test_accuracy             → 0.9999
  test_f1_score             → 0.9999
  test_precision            → 0.9999
  test_recall               → 0.9999

Total metrics logged: 5


,Metric,Value
0,cv_accuracy,0.999860
1,test_accuracy,0.999905
2,test_f1_score,0.999905
3,test_precision,0.999905
4,test_recall,0.999905


In [9]:
# ──────────────────────────────────────────────
# Inspect Parameters
# ──────────────────────────────────────────────
params = run_data.data.params

print("=" * 80)
print("⚙️  PARAMETERS (Best Hyperparameters)")
print("=" * 80)

params_df = pd.DataFrame({
    'Parameter': list(params.keys()),
    'Value': list(params.values())
}).sort_values('Parameter').reset_index(drop=True)

for _, row in params_df.iterrows():
    print(f"  {row['Parameter']:35s} → {row['Value']}")

print(f"\nTotal params logged: {len(params)}")
params_df

⚙️  PARAMETERS (Best Hyperparameters)
  classifier__max_depth               → None
  classifier__min_samples_leaf        → 1
  classifier__min_samples_split       → 2
  classifier__n_estimators            → 50

Total params logged: 4


,Parameter,Value
0,classifier__max_depth,None
1,classifier__min_samples_leaf,1
2,classifier__min_samples_split,2
3,classifier__n_estimators,50


In [10]:
# ──────────────────────────────────────────────
# Inspect Tags
# ──────────────────────────────────────────────
tags = run_data.data.tags

print("=" * 80)
print("🏷️  TAGS")
print("=" * 80)

# Separate custom tags from MLflow system tags
custom_tags = {k: v for k, v in tags.items() if not k.startswith('mlflow.')}
system_tags = {k: v for k, v in tags.items() if k.startswith('mlflow.')}

print("\n── Custom Tags ──")
for key, value in sorted(custom_tags.items()):
    print(f"  {key:25s} → {value}")

print("\n── MLflow System Tags ──")
for key, value in sorted(system_tags.items()):
    print(f"  {key:40s} → {value}")

🏷️  TAGS

── Custom Tags ──
  dataset                   → online_retail
  model_type                → RandomForestClassifier
  pipeline                  → preprocessor + classifier
  tuning_method             → GridSearchCV

── MLflow System Tags ──
  mlflow.runName                           → best_rf_pipeline
  mlflow.source.name                       → 2_Hyperparameter_tuning.ipynb
  mlflow.source.type                       → NOTEBOOK
  mlflow.user                              → jogeswararao


In [11]:
# ──────────────────────────────────────────────
# Check Model Version & Artifact Info
# ──────────────────────────────────────────────
print("=" * 80)
print("📦 MODEL VERSION & ARTIFACT INFO")
print("=" * 80)

# List all artifacts for this run
client = mlflow.tracking.MlflowClient()
artifacts = client.list_artifacts(RUN_ID)

print("\n── Artifacts in this Run ──")
for artifact in artifacts:
    print(f"  📁 {artifact.path} (is_dir: {artifact.is_dir})")
    # List sub-artifacts if it's a directory
    if artifact.is_dir:
        sub_artifacts = client.list_artifacts(RUN_ID, artifact.path)
        for sub in sub_artifacts:
            size_info = f", size: {sub.file_size} bytes" if sub.file_size else ""
            print(f"      📄 {sub.path}{size_info}")

# Get model metadata (MLmodel file info)
print("\n── Model URI ──")
print(f"  runs:/{RUN_ID}/best_rf_pipeline_model")

# Check if model is registered in Model Registry
print("\n── Model Registry Check ──")
try:
    registered_models = client.search_registered_models()
    found = False
    for rm in registered_models:
        for mv in rm.latest_versions:
            if mv.run_id == RUN_ID:
                print(f"  ✅ Model is REGISTERED in Model Registry!")
                print(f"  Model Name:    {rm.name}")
                print(f"  Version:       {mv.version}")
                print(f"  Stage:         {mv.current_stage}")
                print(f"  Status:        {mv.status}")
                print(f"  Creation Time: {pd.Timestamp(mv.creation_timestamp, unit='ms')}")
                found = True
    if not found:
        print("  ℹ️  Model is NOT registered in the Model Registry.")
        print("  To register, use: mlflow.register_model(model_uri, 'model_name')")
except Exception as e:
    print(f"  ℹ️  Model Registry check: {e}")
    print("  Model is tracked via Run ID (not registered in Model Registry).")

📦 MODEL VERSION & ARTIFACT INFO

── Artifacts in this Run ──
  📁 params (is_dir: True)
      📄 params/best_params.csv, size: 243 bytes
  📁 plots (is_dir: True)
      📄 plots/confusion_matrix.png, size: 45315 bytes

── Model URI ──
  runs:/e92368539ff94b94a7f3d6356f2b3759/best_rf_pipeline_model

── Model Registry Check ──
  ℹ️  Model is NOT registered in the Model Registry.
  To register, use: mlflow.register_model(model_uri, 'model_name')


---
## 5️⃣ Verify Loaded Model — Make Predictions

In [12]:
# ──────────────────────────────────────────────
# Create sample data to verify the loaded model
# ──────────────────────────────────────────────

# Sample input matching the training features: Quantity, UnitPrice, Country
sample_data = pd.DataFrame({
    'Quantity': [6, 12, 1, 50, 3],
    'UnitPrice': [2.55, 4.95, 15.00, 1.25, 8.50],
    'Country': ['United Kingdom', 'France', 'Germany', 'United Kingdom', 'Spain']
})

print("── Sample Input Data ──")
display(sample_data)

# Make predictions using the loaded model
predictions = loaded_model.predict(sample_data)

# Add predictions to the sample data
sample_data['Predicted_HighValue'] = predictions
sample_data['Predicted_Label'] = sample_data['Predicted_HighValue'].map({0: 'Low-Value', 1: 'High-Value'})

print("\n── Predictions ──")
display(sample_data)

print("\n✅ Loaded model is working correctly!")

── Sample Input Data ──


,Quantity,UnitPrice,Country
0,6,2.55,United Kingdom
1,12,4.95,France
2,1,15.00,Germany
3,50,1.25,United Kingdom
4,3,8.50,Spain



── Predictions ──


,Quantity,UnitPrice,Country,Predicted_HighValue,Predicted_Label
0,6,2.55,United Kingdom,1,High-Value
1,12,4.95,France,1,High-Value
2,1,15.00,Germany,1,High-Value
3,50,1.25,United Kingdom,1,High-Value
4,3,8.50,Spain,1,High-Value



✅ Loaded model is working correctly!


---
## 6️⃣ Complete Run Summary

In [13]:
# ──────────────────────────────────────────────
# Complete Summary of the Loaded Run
# ──────────────────────────────────────────────
print("\n" + "█" * 80)
print("       📋 COMPLETE RUN SUMMARY")
print("█" * 80)

print(f"\n  🏷️  Experiment:  {EXPERIMENT_NAME}")
print(f"  🔑 Run ID:      {RUN_ID}")
print(f"  📛 Run Name:    {run_data.info.run_name}")
print(f"  ✅ Status:      {run_data.info.status}")

print(f"\n  ── ⚙️  Best Hyperparameters ──")
for k, v in sorted(params.items()):
    print(f"     {k}: {v}")

print(f"\n  ── 📊 Metrics ──")
for k, v in sorted(metrics.items()):
    print(f"     {k}: {v:.4f}")

print(f"\n  ── 📦 Artifacts ──")
for artifact in artifacts:
    print(f"     {artifact.path}/" if artifact.is_dir else f"     {artifact.path}")

print(f"\n  ── 🏷️  Custom Tags ──")
for k, v in sorted(custom_tags.items()):
    print(f"     {k}: {v}")

print(f"\n  ── 🤖 Model Info ──")
print(f"     Model Type: {type(loaded_model.named_steps['classifier']).__name__}")
print(f"     Pipeline Steps: {list(loaded_model.named_steps.keys())}")
print(f"     Model URI: runs:/{RUN_ID}/best_rf_pipeline_model")

print("\n" + "█" * 80)


████████████████████████████████████████████████████████████████████████████████
       📋 COMPLETE RUN SUMMARY
████████████████████████████████████████████████████████████████████████████████

  🏷️  Experiment:  RF model versioning demo
  🔑 Run ID:      e92368539ff94b94a7f3d6356f2b3759
  📛 Run Name:    best_rf_pipeline
  ✅ Status:      FINISHED

  ── ⚙️  Best Hyperparameters ──
     classifier__max_depth: None
     classifier__min_samples_leaf: 1
     classifier__min_samples_split: 2
     classifier__n_estimators: 50

  ── 📊 Metrics ──
     cv_accuracy: 0.9999
     test_accuracy: 0.9999
     test_f1_score: 0.9999
     test_precision: 0.9999
     test_recall: 0.9999

  ── 📦 Artifacts ──
     params/
     plots/

  ── 🏷️  Custom Tags ──
     dataset: online_retail
     model_type: RandomForestClassifier
     pipeline: preprocessor + classifier
     tuning_method: GridSearchCV

  ── 🤖 Model Info ──
     Model Type: RandomForestClassifier
     Pipeline Steps: ['preprocessor', 'classifier'